In [7]:
import os
import gc
import timeit
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.special import expit

from scipy.sparse import csr_matrix
from sklearn.metrics import accuracy_score, recall_score

In [2]:
MODELS_DIR = pathlib.Path("../models/keras")
RESULTS_DIR = pathlib.Path("../results")
DATA_PATH = pathlib.Path("../data/fdia_dataset_processed.npz")

RESULTS_DIR.mkdir(exist_ok=True)

data = np.load(DATA_PATH)

X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"]

X_test_lstm = np.transpose(X_test, (0, 2, 1))

print("X_test:", X_test.shape)
print("X_test LSTM:", X_test_lstm.shape)
print("y_test:", y_test.shape)

X_test: (9720, 6, 83)
X_test LSTM: (9720, 83, 6)
y_test: (9720,)


In [3]:
np.random.seed(42)

sample_size = int(0.10 * len(X_test_lstm))

timing_idx = np.random.choice(
    len(X_test_lstm),
    size=sample_size,
    replace=False
)

X_lstm_timing = X_test_lstm[timing_idx]

print("Timing fraction: 10%")
print("Timing samples:", len(X_lstm_timing))
print("Timing shape:", X_lstm_timing.shape)

Timing fraction: 10%
Timing samples: 972
Timing shape: (972, 83, 6)


# MLP CSR

## MLP dataset

In [4]:
X_mlp_timing = X_test[timing_idx]

X_test_cnn = np.transpose(X_test, (0, 2, 1))
X_cnn_timing = X_test_cnn[timing_idx]

print("MLP timing:", X_mlp_timing.shape)
print("CNN timing:", X_cnn_timing.shape)

MLP timing: (972, 6, 83)
CNN timing: (972, 83, 6)


## MLP Functions


In [8]:
def apply_activation(x, activation):
    if activation == "relu":
        return np.maximum(x, 0)

    if activation == "sigmoid":
        return expit(x)

    return x


def prepare_mlp_csr(model):
    layers = []

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            kernel, bias = layer.get_weights()

            layers.append({
                "kernel": csr_matrix(kernel.astype(np.float32)),
                "bias": bias.astype(np.float32),
                "activation": layer.activation.__name__
            })

    return layers


def predict_mlp_csr(X, layers):
    x = X.reshape(X.shape[0], -1)

    for layer in layers:
        x = layer["kernel"].T.dot(x.T).T + layer["bias"]
        x = apply_activation(x, layer["activation"])

    return x


def get_mlp_csr_size_kb(layers):
    size_bytes = 0

    for layer in layers:
        matrix = layer["kernel"]

        size_bytes += matrix.data.nbytes
        size_bytes += matrix.indices.nbytes
        size_bytes += matrix.indptr.nbytes
        size_bytes += layer["bias"].nbytes

    return size_bytes / 1024

## MLP CSR Test

In [9]:
mlp_csr_models = {
    "MLP_Weight_CSR": "MLP_WeightPruned.keras",
    "MLP_WeightNode_CSR": "MLP_WeightNodePruned.keras",
    "MLP_NodeWeight_CSR": "MLP_NodeWeightPruned.keras"
}

mlp_csr_results = []

for name, filename in mlp_csr_models.items():
    print(f"\nTesting: {name}")

    model = tf.keras.models.load_model(MODELS_DIR / filename, compile=False)
    layers = prepare_mlp_csr(model)

    # Metrics
    predictions = predict_mlp_csr(X_test, layers)
    y_pred = (predictions >= 0.5).astype(int).flatten()

    accuracy = accuracy_score(y_test, y_pred)
    fdia_recall = recall_score(y_test, y_pred, pos_label=1)
    fault_recall = recall_score(y_test, y_pred, pos_label=0)

    # CSR storage
    size_kb = get_mlp_csr_size_kb(layers)

    # Warm-up
    predict_mlp_csr(X_mlp_timing, layers)

    # One-shot timing
    run_times = []

    for _ in range(10):
        start = timeit.default_timer()

        predict_mlp_csr(X_mlp_timing, layers)

        end = timeit.default_timer()

        run_times.append(((end - start) * 1000) / len(X_mlp_timing))

    inference_time = np.mean(run_times)

    result = {
        "architecture": "MLP",
        "model": name,
        "parameters": model.count_params(),
        "accuracy": accuracy,
        "fdia_recall": fdia_recall,
        "fault_recall": fault_recall,
        "model_size_kb": size_kb,
        "inference_time_ms": inference_time
    }

    mlp_csr_results.append(result)

    print(f"Accuracy:       {accuracy:.6f}")
    print(f"FDIA Recall:    {fdia_recall:.6f}")
    print(f"Fault Recall:   {fault_recall:.6f}")
    print(f"CSR Size:       {size_kb:.3f} KB")
    print(f"Inference Time: {inference_time:.6f} ms/sample")


Testing: MLP_Weight_CSR
Accuracy:       0.998148
FDIA Recall:    0.998264
Fault Recall:   0.998044
CSR Size:       139.922 KB
Inference Time: 0.009617 ms/sample

Testing: MLP_WeightNode_CSR
Accuracy:       0.986214
FDIA Recall:    0.971354
Fault Recall:   0.999609
CSR Size:       95.570 KB
Inference Time: 0.007687 ms/sample

Testing: MLP_NodeWeight_CSR
Accuracy:       0.997737
FDIA Recall:    0.997613
Fault Recall:   0.997848
CSR Size:       91.305 KB
Inference Time: 0.006468 ms/sample


# CNN CSR

## CNN CSR Functions


In [11]:
def prepare_cnn_csr(model):
    layers = []

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv1D):
            kernel, bias = layer.get_weights()

            # Conv1D kernel: (kernel_size, input_channels, filters)
            kernel_2d = kernel.reshape(-1, kernel.shape[-1])

            layers.append({
                "type": "conv",
                "kernel": csr_matrix(kernel_2d.astype(np.float32)),
                "bias": bias.astype(np.float32)
            })

        elif isinstance(layer, tf.keras.layers.Dense):
            kernel, bias = layer.get_weights()

            layers.append({
                "type": "dense",
                "kernel": kernel.astype(np.float32),
                "bias": bias.astype(np.float32)
            })

    return layers

def get_cnn_csr_size_kb(layers):
    size_bytes = 0

    for layer in layers:
        if layer["type"] == "conv":
            matrix = layer["kernel"]

            size_bytes += matrix.data.nbytes
            size_bytes += matrix.indices.nbytes
            size_bytes += matrix.indptr.nbytes
            size_bytes += layer["bias"].nbytes

        elif layer["type"] == "dense":
            size_bytes += layer["kernel"].nbytes
            size_bytes += layer["bias"].nbytes

    return size_bytes / 1024

## CNN CSR Test

In [12]:
cnn_csr_models = {
    "CNN1D_Weight_CSR": "CNN1D_WeightPruned.keras",
    "CNN1D_WeightNode_CSR": "CNN1D_WeightNodePruned.keras",
    "CNN1D_NodeWeight_CSR": "CNN1D_NodeWeightPruned.keras"
}

for name, filename in cnn_csr_models.items():
    model = tf.keras.models.load_model(MODELS_DIR / filename, compile=False)

    layers = prepare_cnn_csr(model)
    csr_size_kb = get_cnn_csr_size_kb(layers)

    print(f"{name}: {csr_size_kb:.3f} KB")

CNN1D_Weight_CSR: 283.875 KB
CNN1D_WeightNode_CSR: 179.621 KB
CNN1D_NodeWeight_CSR: 146.000 KB
